In [6]:
import os
import re
import json
import unicodedata
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

load_dotenv(override=True)
api_key = os.getenv("GAPGPT_API_KEY")
base_url = os.getenv("GAPGPT_BASE_URL", "https://api.gapgpt.app/v1")
model_name = os.getenv("GAPGPT_MODEL", "gpt-4o-mini")

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
    timeout=60,
    max_retries=2
)

RISK_TO_ID = {
    "مناسب": 0,
    "کم‌ریسک": 1,
    "پرریسک": 2
}
COLUMNS = ["text", "label", "risk_name", "category", "risk_factor", "seed_id"]

seed_clauses = [
    {"category": "payment", "text": "پرداخت حق‌الزحمه پس از تحویل و تأیید نهایی پروژه ظرف مدت ۳ روز کاری انجام می‌شود."},
    {"category": "payment", "text": "تسویه حساب منوط به رضایت کامل و بدون قید و شرط کارفرما از تمام ابعاد پروژه است."},
    {"category": "scope", "text": "شرح خدمات و خروجی‌های مورد انتظار در پیوست قرارداد مشخص شده و تغییر آن مستلزم الحاقیه است."},
    {"category": "change_request", "text": "فریلنسر موظف است هرگونه تغییرات مدنظر کارفرما را تا پایان پروژه بدون هزینه اضافی اعمال نماید."},
    {"category": "termination", "text": "هر یک از طرفین با اخطار کتبی ۷ روزه و تسویه کارهای انجام‌شده حق فسخ قرارداد را دارند."},
    {"category": "termination", "text": "کارفرما در هر مرحله می‌تواند بدون ذکر دلیل قرارداد را فسخ کند و صرفاً نیمی از هزینه را بپردازد."},
    {"category": "liability", "text": "جبران خسارات احتمالی ناشی از خطای نرم‌افزاری حداکثر تا سقف مبلغ کل قرارداد خواهد بود."},
    {"category": "intellectual_property", "text": "کلیه حقوق مادی و معنوی کدها پس از تسویه حساب کامل به کارفرما منتقل می‌گردد."}
]

# تابع بهبود یافته و مقاوم در برابر انواع فاصله و نیم‌فاصله
def normalize_risk_name(value):
    if not value:
        return None
    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = text.replace("ي", "ی").replace("ك", "ک")
    # حذف تمام فاصله‌ها و نیم‌فاصله‌ها
    compact = re.sub(r"[\s\u200c\-_–—]+", "", text)
    
    if compact in ["مناسب", "امن", "safe", "safe_risk"]:
        return "مناسب"
    if compact in ["کمریسک", "ریسککم", "lowrisk", "low"]:
        return "کم‌ریسک"
    if compact in ["پرریسک", "ریسکپر", "ریسکبالا", "highrisk", "high", "خطرناک"]:
        return "پرریسک"
    return None

def parse_json_response(content):
    if not content:
        raise ValueError("پاسخ دریافتی خالی است.")
    text = str(content).strip()
    fence = "\x60\x60\x60"
    if text.startswith(fence):
        text = text[3:]
        if text.lower().startswith("json"):
            text = text[4:]
    if text.endswith(fence):
        text = text[:-3]
    text = text.strip()
    
    try:
        return json.loads(text)
    except Exception:
        pass
        
    start_obj = text.find("{")
    end_obj = text.rfind("}")
    if start_obj != -1 and end_obj > start_obj:
        try:
            return json.loads(text[start_obj:end_obj + 1])
        except Exception:
            pass
            
    raise ValueError("پارس JSON ناموفق بود: " + text[:150])

dataset_records = []

print("در حال تولید داده‌ها با نرمالایزر جدید...")
for seed_id, seed in enumerate(tqdm(seed_clauses, desc="Generating"), start=1):
    prompt = f"""
بر اساس بند پایه زیر، ۳ بند مستقل و کاربردی قرارداد فریلنسری با سه سطح ریسک به فارسی تولید کن:
۱. «مناسب»: تعهدات شفاف، دوطرفه و سازوکار عادلانه.
۲. «کم‌ریسک»: قابل قبول، اما دارای یک ابهام زمانی یا اجرایی کوچک.
۳. «پرریسک»: تعهد یک‌طرفه، سلب حق، یا ابهام جدی در پرداخت و تایید کار.

دسته‌بندی: {seed['category']}
بند پایه: {seed['text']}

خروجی صرفاً یک شیء JSON با ساختار زیر:
{{
  "clauses": [
    {{
      "category": "{seed['category']}",
      "text": "متن بند مناسب",
      "risk_name": "مناسب",
      "risk_factor": "توضیح مختصر ریسک"
    }},
    {{
      "category": "{seed['category']}",
      "text": "متن بند کم‌ریسک",
      "risk_name": "کم‌ریسک",
      "risk_factor": "توضیح مختصر ریسک"
    }},
    {{
      "category": "{seed['category']}",
      "text": "متن بند پرریسک",
      "risk_name": "پرریسک",
      "risk_factor": "توضیح مختصر ریسک"
    }}
  ]
}}
"""
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a legal clause synthesizer. Always respond with pure valid JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        data = parse_json_response(response.choices[0].message.content)
        clauses = data.get("clauses", []) if isinstance(data, dict) else []
        
        for item in clauses:
            t = str(item.get("text", "")).strip()
            r_name = normalize_risk_name(item.get("risk_name"))
            if t and r_name in RISK_TO_ID:
                dataset_records.append({
                    "text": t,
                    "label": RISK_TO_ID[r_name],
                    "risk_name": r_name,
                    "category": item.get("category", seed["category"]),
                    "risk_factor": item.get("risk_factor", ""),
                    "seed_id": seed_id
                })
    except Exception as e:
        print(f"\nخطا در Seed شماره {seed_id}: {e}")

df = pd.DataFrame(dataset_records, columns=COLUMNS)
print(f"\nتعداد کل رکوردهای تولید شده: {len(df)}")

if not df.empty:
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print("\nتوزیع کلاس‌ها:")
    print(df["risk_name"].value_counts())
    
    df.to_json("contract_risk_full.jsonl", orient="records", lines=True, force_ascii=False)
    df[["text", "label"]].to_json("contract_risk_train.jsonl", orient="records", lines=True, force_ascii=False)
    print("\nفایل‌ها با هر ۳ کلاس با موفقیت بازنویسی شدند.")


در حال تولید داده‌ها با نرمالایزر جدید...


Generating: 100%|██████████| 8/8 [04:50<00:00, 36.29s/it]


تعداد کل رکوردهای تولید شده: 24

توزیع کلاس‌ها:
risk_name
مناسب      8
کم‌ریسک    8
پرریسک     8
Name: count, dtype: int64

فایل‌ها با هر ۳ کلاس با موفقیت بازنویسی شدند.


In [7]:
import pandas as pd
df_full = pd.read_json("contract_risk_full.jsonl", lines=True)
print("کلاس‌های ذخیره شده:")
print(df_full["risk_name"].value_counts())
print("\nتعداد کل:", len(df_full))


کلاس‌های ذخیره شده:
risk_name
مناسب      8
کم‌ریسک    8
پرریسک     8
Name: count, dtype: int64

تعداد کل: 24
